# Movie Recommendation System
## Phase 6: User-Based Collaborative Filtering

In this notebook, we will build a User-Based Collaborative
Filtering recommendation engine.

The system will identify users with similar rating patterns
and recommend movies based on the preferences of similar users.

We will use mean-centered ratings and cosine similarity
to identify similar users.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_DIR = Path("../data/processed")

ratings = pd.read_csv(DATA_DIR / "ratings_clean.csv")
movies = pd.read_csv(DATA_DIR / "movies_clean.csv")

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [3]:
user_movie_matrix = ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

print("Rating matrix shape:", user_movie_matrix.shape)

Rating matrix shape: (6040, 3706)


In [4]:
user_means = ratings.groupby("UserID")["Rating"].mean()

print(user_means.head())

UserID
1    4.188679
2    3.713178
3    3.901961
4    4.190476
5    3.146465
Name: Rating, dtype: float64


In [5]:
ratings_centered = ratings.copy()

ratings_centered["UserMean"] = ratings_centered["UserID"].map(
    user_means
)

ratings_centered["CenteredRating"] = (
    ratings_centered["Rating"] - ratings_centered["UserMean"]
)

ratings_centered.head()

,UserID,MovieID,Rating,Timestamp,RatingDate,UserMean,CenteredRating
0,1,1193,5.0,978300760,2000-12-31 22:12:40,4.188679,0.811321
1,1,661,3.0,978302109,2000-12-31 22:35:09,4.188679,-1.188679
2,1,914,3.0,978301968,2000-12-31 22:32:48,4.188679,-1.188679
3,1,3408,4.0,978300275,2000-12-31 22:04:35,4.188679,-0.188679
4,1,2355,5.0,978824291,2001-01-06 23:38:11,4.188679,0.811321


In [6]:
user_ids = user_movie_matrix.index.tolist()
movie_ids = user_movie_matrix.columns.tolist()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}

row_indices = ratings_centered["UserID"].map(
    user_to_index
).to_numpy()

column_indices = ratings_centered["MovieID"].map(
    movie_to_index
).to_numpy()

values = ratings_centered["CenteredRating"].to_numpy(
    dtype=np.float32
)

centered_matrix = csr_matrix(
    (values, (row_indices, column_indices)),
    shape=(len(user_ids), len(movie_ids))
)

print("Sparse matrix shape:", centered_matrix.shape)
print("Stored values:", centered_matrix.nnz)

Sparse matrix shape: (6040, 3706)
Stored values: 1000209


In [7]:
user_similarity = cosine_similarity(
    centered_matrix,
    dense_output=False
)

print("User similarity matrix shape:", user_similarity.shape)

User similarity matrix shape: (6040, 6040)


In [8]:
def get_similar_users(user_id, top_n=10):

    if user_id not in user_to_index:
        return "User ID not found."

    user_index = user_to_index[user_id]

    similarity_scores = user_similarity.getrow(user_index)

    similar_indices = similarity_scores.indices
    similar_scores = similarity_scores.data

    # Exclude the selected user and non-positive similarities
    valid = (
        (similar_indices != user_index) &
        (similar_scores > 0)
    )

    similar_indices = similar_indices[valid]
    similar_scores = similar_scores[valid]

    if len(similar_indices) == 0:
        return "No similar users found."

    # Select the highest similarity scores
    top_positions = np.argsort(similar_scores)[::-1][:top_n]

    selected_indices = similar_indices[top_positions]
    selected_scores = similar_scores[top_positions]

    result = pd.DataFrame({
        "UserID": [user_ids[i] for i in selected_indices],
        "SimilarityScore": selected_scores
    })

    return result

In [9]:
get_similar_users(user_id=1, top_n=10)

,UserID,SimilarityScore
0,1337,0.189242
1,379,0.159893
2,5404,0.155154
3,49,0.148455
4,2607,0.148105
5,5739,0.146056
6,5635,0.144795
7,2495,0.139092
8,4159,0.138192
9,4724,0.135727


In [10]:
def recommend_movies(user_id, top_n=10, neighbor_count=20):

    if user_id not in user_to_index:
        return "User ID not found."

    # Find similar users
    similar_users = get_similar_users(
        user_id,
        top_n=neighbor_count
    )

    if isinstance(similar_users, str):
        return similar_users

    # Map similar user IDs to their similarity scores
    similarity_map = dict(
        zip(
            similar_users["UserID"],
            similar_users["SimilarityScore"]
        )
    )

    # Get ratings from similar users
    neighbor_ratings = ratings[
        ratings["UserID"].isin(similarity_map.keys())
    ].copy()

    # Calculate weighted rating deviations
    neighbor_ratings["Similarity"] = (
        neighbor_ratings["UserID"].map(similarity_map)
    )

    neighbor_ratings["UserMean"] = (
        neighbor_ratings["UserID"].map(user_means)
    )

    neighbor_ratings["WeightedDeviation"] = (
        neighbor_ratings["Similarity"] *
        (neighbor_ratings["Rating"] - neighbor_ratings["UserMean"])
    )

    # Calculate predicted ratings for each movie
    predictions = neighbor_ratings.groupby("MovieID").agg(
        WeightedSum=("WeightedDeviation", "sum"),
        WeightSum=("Similarity", lambda x: x.abs().sum())
    )

    predictions = predictions[
        predictions["WeightSum"] > 0
    ]

    predictions["PredictedRating"] = (
        user_means[user_id] +
        predictions["WeightedSum"] / predictions["WeightSum"]
    )

    # Remove movies already rated by the target user
    watched_movies = set(
        ratings.loc[
            ratings["UserID"] == user_id,
            "MovieID"
        ]
    )

    predictions = predictions.drop(
        index=predictions.index.intersection(watched_movies)
    )

    # Select highest predicted ratings
    predictions = predictions.sort_values(
        "PredictedRating",
        ascending=False
    ).head(top_n)

    # Attach movie titles and genres
    recommendations = predictions.merge(
        movies,
        on="MovieID",
        how="left"
    )

    return recommendations[
        ["MovieID", "Title", "Genres", "PredictedRating"]
    ].reset_index(drop=True)

In [11]:
recommend_movies(user_id=1, top_n=10)

,MovieID,Title,Genres,PredictedRating
0,2,Jumanji (1995),Adventure|Children's|Fantasy,5.747273
1,3478,"Bamba, La (1987)",Drama,5.747273
2,3489,Hook (1991),Adventure|Fantasy,5.747273
3,3448,"Good Morning, Vietnam (1987)",Comedy|Drama|War,5.747273
4,3608,Pee-wee's Big Adventure (1985),Comedy,5.747273
5,292,Outbreak (1995),Action|Drama|Thriller,5.747273
6,253,Interview with the Vampire (1994),Drama|Horror,5.747273
7,3138,Stealing Home (1988),Drama,5.747273
8,3098,"Natural, The (1984)",Drama,5.747273
9,2872,Excalibur (1981),Action|Drama|Fantasy|Romance,5.747273


In [12]:
recommend_movies(user_id=10, top_n=10)

,MovieID,Title,Genres,PredictedRating
0,147,"Basketball Diaries, The (1995)",Drama,5.925448
1,215,Before Sunrise (1995),Drama|Romance,5.925448
2,428,"Bronx Tale, A (1993)",Drama,5.925448
3,328,Tales From the Crypt Presents: Demon Knight (1...,Horror,5.925448
4,337,What's Eating Gilbert Grape (1993),Drama,5.925448
5,1061,Sleepers (1996),Crime|Drama,5.925448
6,611,Hellraiser: Bloodline (1996),Action|Horror|Sci-Fi,5.925448
7,879,"Relic, The (1997)",Horror,5.925448
8,1696,Bent (1997),Drama|War,5.862630
9,2171,"Next Stop, Wonderland (1998)",Comedy|Drama|Romance,5.862630


In [13]:
recommend_movies(user_id=1, top_n=5)

,MovieID,Title,Genres,PredictedRating
0,2,Jumanji (1995),Adventure|Children's|Fantasy,5.747273
1,3478,"Bamba, La (1987)",Drama,5.747273
2,3489,Hook (1991),Adventure|Fantasy,5.747273
3,3448,"Good Morning, Vietnam (1987)",Comedy|Drama|War,5.747273
4,3608,Pee-wee's Big Adventure (1985),Comedy,5.747273
